# Solutions - Assignment 2: Gathering data from an API

## Instructor reference

This notebook is a complete worked solution to [`topic19_assignment_api_gathering.ipynb`](../../data_analysis_process/assignments/topic19_assignment_api_gathering.ipynb).

**It solves Option A, the World Bank Indicators API**, gathering life expectancy at birth for Ghana, Kenya, and Senegal from 2000 to 2023. Option A was chosen for this reference because it exercises the two hardest ideas in the assignment:

- its response is a **two-item list**, `[metadata, records]`, rather than a plain array, so a student who assumed the obvious shape gets an immediate error;
- a wrong indicator code returns **HTTP 200 with an error message in the body**, which is the trap Part 4 is built around.

Students who chose Option B (Frankfurter) or Option C (USGS) should have the same structure with a different `flatten_response()`. Notes for those two options appear at the relevant parts.

### Part 0 is not reproduced

The tutorial cells in the assignment are already complete and correct. Only the students' five tutorial answers are given here, in the next cell.

### Running this notebook

It makes several requests to `api.worldbank.org`, including three deliberately broken ones in Part 4, and needs an internet connection. The `requests` sections are guarded so it still runs where `requests` is not installed.

## Part 0 - Tutorial answers

**1. Sorting one array without the others.** The `daily` dictionary links each date to its temperature purely by **position** - the third temperature belongs to the third date only because both are third. Sorting `temperature_2m_max` alone would re-order the temperatures while the dates stayed put, silently attaching every reading to the wrong day.

The chart would still look believable because the *set* of values would be unchanged: the same range, the same distribution, a plausible-looking seasonal curve. Nothing would be missing or out of range. Only the pairing would be wrong, and no summary statistic would reveal it. This is why the response is converted into a DataFrame immediately: once the pairing is a row, it cannot be broken by sorting.

**2. Why save the raw response first.** Because it cannot be re-requested with any guarantee. The response is generated when you ask; it is not a file sitting on a server. Open-Meteo could revise its model, change the endpoint, restrict access, or be unreachable when a marker tries to check the work. The saved file is the only evidence of what was actually received and analysed. Saving it *before* parsing also means that a bug in the parsing code cannot cost you the data.

**3. Summed rainfall, averaged temperature.** Rainfall is a quantity that accumulates: 5 mm on Monday plus 5 mm on Tuesday is 10 mm of rain that fell, and the total is the meaningful figure. Temperature is a state, not an accumulation - there is no such thing as having "collected" 900 degrees over a month.

A chart of summed monthly temperature would essentially be a chart of **how many days are in each month**, scaled by the temperature. February would appear coolest simply for being short, and a leap year would change the answer. The aggregation you choose is a statement about what the number means.

**4. The shifted coordinates.** The reanalysis model works on a fixed grid, so the API returns data for the grid cell nearest the requested point rather than for the exact coordinates. The response reports the cell's actual centre, which is how you know.

It matters whenever the distance between the requested point and the grid cell is large relative to the variation you are trying to measure: comparing two neighbouring towns that fall in the same cell would produce two identical series and an apparent finding of "no difference" that is really an artefact of resolution. It also matters for elevation - a coastal city can land in a cell whose reported elevation is not its own.

**5. Something only the documentation could tell us.** That `temperature_2m_max` is the air temperature **2 metres above the ground**, the standard height for weather observation, rather than ground-surface or satellite temperature. Also acceptable: that the archive lags real time by about five days; that the values are model reanalysis rather than station readings; or that the free tier is limited to non-commercial use. None of these is discoverable from the numbers, and each changes what can honestly be claimed.

> **Marking note - Part 0.** Questions 1 and 3 are the ones that separate understanding from transcription. For question 1, the key insight is that the chart would look *fine* - a student who only says "the data would be wrong" has missed the point, which is that this class of error is invisible to inspection. For question 3, look for the recognition that summed temperature would mostly measure month length.

## Setup

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
from urllib.parse import urlencode
from urllib.request import Request, urlopen
from urllib.error import HTTPError, URLError
import json
import sys

import pandas as pd
import matplotlib.pyplot as plt

try:
    import requests
    REQUESTS_AVAILABLE = True
except ImportError:
    REQUESTS_AVAILABLE = False

BASE_DIR = Path.cwd()
DATA_DIR = BASE_DIR / "data"
RAW_DIR = DATA_DIR / "raw"
CLEAN_DIR = DATA_DIR / "clean"
CHART_DIR = BASE_DIR / "charts"

for folder in (RAW_DIR, CLEAN_DIR, CHART_DIR):
    folder.mkdir(parents=True, exist_ok=True)

pd.set_option("display.max_columns", 25)
pd.set_option("display.width", 120)
plt.rcParams["figure.figsize"] = (11, 5)
plt.rcParams["figure.dpi"] = 110

USER_AGENT = "ITFY-Ghana-student-assignment/1.0"
DEFAULT_HEADERS = {"Accept": "application/json", "User-Agent": USER_AGENT}

print("pandas version:     ", pd.__version__)
print("requests available: ", REQUESTS_AVAILABLE)

## Part 1 - Choose your API and write your question

**TODOs 1 to 5.**

In [ ]:
API_OPTIONS = {
    "A": {
        "name": "World Bank Indicators API",
        "endpoint": "https://api.worldbank.org/v2/country/GHA;KEN;SEN/indicator/SP.DYN.LE00.IN",
        "documentation": "https://datahelpdesk.worldbank.org/knowledgebase/articles/889392",
        "params": {"date": "2000:2023", "format": "json", "per_page": 500},
        "raw_filename": "world_bank_indicator_raw.json",
        "response_shape": "two-item list: [metadata, records]",
        "record_meaning": "one country, one indicator, one year",
        "api_key_required": False,
    },
    "B": {
        "name": "Frankfurter foreign exchange rates",
        "endpoint": "https://api.frankfurter.dev/v1/2023-01-01..2023-12-31",
        "documentation": "https://frankfurter.dev",
        "params": {"base": "USD", "symbols": "EUR,GBP,JPY,ZAR"},
        "raw_filename": "frankfurter_rates_raw.json",
        "response_shape": "dictionary of dictionaries, keyed by date",
        "record_meaning": "one currency rate against the base on one date",
        "api_key_required": False,
    },
    "C": {
        "name": "USGS Earthquake catalogue",
        "endpoint": "https://earthquake.usgs.gov/fdsnws/event/1/query",
        "documentation": "https://earthquake.usgs.gov/fdsnws/event/1/",
        "params": {"format": "geojson", "starttime": "2023-01-01",
                   "endtime": "2023-12-31", "minmagnitude": 5},
        "raw_filename": "usgs_earthquakes_raw.json",
        "response_shape": "GeoJSON: a list of features with nested properties",
        "record_meaning": "one recorded earthquake",
        "api_key_required": False,
    },
}

# TODO 1
CHOSEN_API = "A"

# TODO 2
RESEARCH_QUESTION = ("How has life expectancy at birth changed in Ghana, Kenya, and "
                     "Senegal between 2000 and 2023, and which of the three gained most?")

# TODO 3 - adapted, not copied: a longer per_page so the whole result fits on one
# page, and an explicit date range chosen to match the question.
INDICATOR_CODE = "SP.DYN.LE00.IN"
INDICATOR_LABEL = "Life expectancy at birth (years)"
COUNTRY_CODES = "GHA;KEN;SEN"
MY_PARAMS = {"date": "2000:2023", "format": "json", "per_page": 500}

# TODO 4 - for the World Bank the country codes and the indicator live in the
# PATH, not the query string, so the endpoint is built from them.
MY_ENDPOINT = (f"https://api.worldbank.org/v2/country/{COUNTRY_CODES}"
               f"/indicator/{INDICATOR_CODE}")

# TODO 5
EXPECTATION = ("I expect all three to have risen since 2000, and I expect Kenya to "
               "show the largest gain because it starts from a low point caused by "
               "the HIV/AIDS epidemic and had the most ground to recover.")

if CHOSEN_API not in API_OPTIONS:
    raise ValueError('Set CHOSEN_API to "A", "B", or "C" before continuing.')

API = API_OPTIONS[CHOSEN_API]
MY_RAW_PATH = RAW_DIR / API["raw_filename"]

print("API:            ", API["name"])
print("Response shape: ", API["response_shape"])
print("My endpoint:    ", MY_ENDPOINT)
print("My parameters:  ", MY_PARAMS)
print("My question:    ", RESEARCH_QUESTION)

> **Marking note - Part 1.** TODO 3 says the parameters must be **adapted**. A student who submitted the example endpoint and parameters completely unchanged has not chosen a slice of the data, and should lose those marks even though the code runs. Any genuine change is acceptable: different countries, a different indicator, a different date range.
>
> Note that for Option A the country codes and indicator are part of the **URL path**, not the query string. Students who tried to pass `country=GHA` as a parameter will have received data for every country in the world, or an error. That confusion is worth a comment rather than a deduction - it is a reasonable mistake and the fix teaches the difference between path and query.

## Part 2 - Build your request URL

**TODOs 6 and 7.**

In [ ]:
def build_api_url(endpoint, params=None):
    """Join an endpoint and a parameter dictionary into a complete URL."""
    # TODO 6 - the empty case must not leave a trailing "?".
    if not params:
        return endpoint
    return f"{endpoint}?{urlencode(params)}"

In [ ]:
# TODO 7
my_url = build_api_url(MY_ENDPOINT, MY_PARAMS)
print("Request URL:")
print(my_url)

print("\nChecks before sending:")
print("  every parameter present:",
      all(str(key) in my_url for key in MY_PARAMS))
print("  format=json present:    ", "format=json" in my_url,
      "  <- without this the World Bank returns XML")
print("  semicolons in the path:  ", ";" in my_url,
      "  <- country codes are in the path, so they are NOT escaped")

# The empty case, which is easy to get wrong.
print("\nEmpty-parameter case:", build_api_url("https://api.worldbank.org/v2/country"))
print("  ends with '?':", build_api_url("https://api.worldbank.org/v2/country").endswith("?"),
      "<- must be False")

> **Marking note - Part 2.** The failure mode here is `f"{endpoint}?{urlencode(params or {})}"`, which returns `https://example.com?` when there are no parameters. Most servers tolerate the stray `?`, so the student's notebook still works and the bug is invisible. TODO 6 asks for it explicitly, so check the empty case rather than only the output.

## Part 3 - Send the request and preserve the raw response

**TODOs 8 to 16.**

In [ ]:
def fetch_json_urllib(endpoint, params=None, raw_path=None, timeout=30):
    """Request JSON with the standard library, saving the raw response first."""
    # TODO 8
    url = build_api_url(endpoint, params)

    # TODO 9 - HTTPError means the server refused; URLError means it was never
    # reached. Keeping them apart is what makes a failure diagnosable.
    request = Request(url, headers=DEFAULT_HEADERS)
    try:
        with urlopen(request, timeout=timeout) as response:
            status = response.status
            raw_text = response.read().decode("utf-8")
    except HTTPError as error:
        raise RuntimeError(
            f"The server refused {url} with HTTP status {error.code}"
        ) from error
    except URLError as error:
        raise RuntimeError(
            f"Could not reach the server for {url}: {error.reason}"
        ) from error

    # TODO 10 - save BEFORE parsing, so a parsing bug cannot lose the data.
    if raw_path is not None:
        Path(raw_path).write_text(raw_text, encoding="utf-8")

    # TODO 11
    payload = json.loads(raw_text)
    request_details = {
        "tool": "urllib",
        "url": url,
        "params": dict(params or {}),
        "status": status,
        "bytes": len(raw_text.encode("utf-8")),
        "retrieved_utc": datetime.now(timezone.utc).isoformat(),
        "raw_path": str(raw_path) if raw_path else None,
    }
    return payload, request_details


def fetch_json_requests(endpoint, params=None, raw_path=None, timeout=30):
    """Request JSON with requests, saving the raw response first."""
    if not REQUESTS_AVAILABLE:
        raise RuntimeError("requests is not installed. Run: pip install requests")

    # TODO 12 - requests builds the query string from params itself.
    response = requests.get(endpoint, params=params, headers=DEFAULT_HEADERS,
                            timeout=timeout)

    # TODO 13
    response.raise_for_status()
    if raw_path is not None:
        Path(raw_path).write_text(response.text, encoding="utf-8")

    request_details = {
        "tool": "requests",
        "url": response.url,
        "params": dict(params or {}),
        "status": response.status_code,
        "bytes": len(response.text.encode("utf-8")),
        "retrieved_utc": datetime.now(timezone.utc).isoformat(),
        "raw_path": str(raw_path) if raw_path else None,
    }
    return response.json(), request_details

In [ ]:
# TODO 14
payload, request_details = fetch_json_urllib(MY_ENDPOINT, MY_PARAMS, MY_RAW_PATH)

# TODO 15
print(json.dumps(request_details, indent=2))
print("\nRaw file exists:", MY_RAW_PATH.exists())
print("Raw file size:  ", f"{MY_RAW_PATH.stat().st_size:,} bytes")

# TODO 16
if REQUESTS_AVAILABLE:
    requests_raw_path = RAW_DIR / ("requests_" + API["raw_filename"])
    payload_via_requests, requests_details = fetch_json_requests(
        MY_ENDPOINT, MY_PARAMS, requests_raw_path)
    print("\nBoth tools returned identical payloads:",
          payload == payload_via_requests)
else:
    print("\nrequests not installed - skipping the comparison.")

> **Marking note - Part 3.** The graded behaviour is **saving before parsing**. A student whose function parses first and saves afterwards loses the data whenever the response is not valid JSON - which is exactly the case they most need it for. Check the order of the lines, not just that a file exists.
>
> Also check the function **returns both** the payload and the details. Students frequently return only the payload and then rebuild the parameters by hand in Part 11, which is how manifests come to disagree with what was actually requested.

## Part 4 - Make it fail on purpose

**TODOs 17 to 21.** This is the most important part of the assignment and carries 14 marks.

In [ ]:
# TODO 17 - failure 1: a host that does not exist.
print("FAILURE 1 - wrong host")
bad_host = "https://api.worldbank.invalid/v2/country/GHA/indicator/SP.DYN.LE00.IN"
try:
    fetch_json_urllib(bad_host, {"format": "json"})
except RuntimeError as error:
    print("  Exception type:", type(error).__name__)
    print("  Message:       ", error)
    print("  Underlying:    ", type(error.__cause__).__name__)
    print("  -> No HTTP status at all. We never reached a server, so there was")
    print("     nothing to return one. This is the URLError branch.")

In [ ]:
# TODO 18 - failure 2: a real host, a path that does not exist.
print("FAILURE 2 - wrong path")
bad_path = "https://api.worldbank.org/v2/country/GHA/indicator/SP.DYN.LE00.IN/nonsense"
try:
    payload_bad_path, _ = fetch_json_urllib(bad_path, {"format": "json"})
    print("  No exception raised. Status was 200.")
    print("  Body received:", json.dumps(payload_bad_path)[:300])
    print("  -> Note: the World Bank answers many malformed paths with a 200 and")
    print("     an error document rather than a 404, which is the same trap as")
    print("     failure 3. Whether you get 404 or 200 here depends on how the")
    print("     path is malformed - try adding a stray segment versus mistyping")
    print("     the /v2/ prefix and compare.")
except RuntimeError as error:
    print("  Exception type:", type(error).__name__)
    print("  Message:       ", error)
    print("  Underlying:    ", type(error.__cause__).__name__,
          "with code", getattr(error.__cause__, "code", "n/a"))
    print("  -> The server WAS reached and refused us. This is the HTTPError")
    print("     branch, and it is a genuinely different problem from failure 1.")

In [ ]:
# TODO 19 - failure 3: a valid request for something that does not exist.
# Deliberately NOT wrapped in try/except, because nothing is raised.
print("FAILURE 3 - wrong parameter (a made-up indicator code)")
bad_indicator = "https://api.worldbank.org/v2/country/GHA/indicator/NOT.A.REAL.CODE"
bad_payload, bad_details = fetch_json_urllib(bad_indicator, {"format": "json"})

print("  HTTP status:", bad_details["status"], " <- success!")
print("  Body:", json.dumps(bad_payload, indent=2)[:400])
print()
print("  -> This is the trap. The request succeeded at the HTTP level. The")
print("     server understood us perfectly and returned a valid JSON document")
print("     whose CONTENT is an error message. Code that trusts the status code")
print("     would carry on and fail much later, somewhere unrelated, with a")
print("     confusing message about a missing key or an index out of range.")

In [ ]:
def check_api_payload(payload, api_option):
    """Raise a clear error if a parsed payload is not the data we asked for."""
    # TODO 20
    if api_option == "A":
        # A good World Bank response is [metadata, records].
        if not isinstance(payload, list):
            raise ValueError(
                f"Expected a list from the World Bank API, got {type(payload).__name__}. "
                "If this is a dict, you probably omitted format=json and received XML "
                "or an error document."
            )
        if payload and isinstance(payload[0], dict) and "message" in payload[0]:
            raise ValueError(
                f"The World Bank returned an API error: {payload[0]['message']}"
            )
        if len(payload) < 2:
            raise ValueError(
                f"Expected [metadata, records] but got a list of {len(payload)} item(s)."
            )
        if not payload[1]:
            raise ValueError(
                "The request was valid but matched no data at all. Check the country "
                "codes, the indicator code, and the date range."
            )

    elif api_option == "B":
        if not isinstance(payload, dict):
            raise ValueError(f"Expected a dict, got {type(payload).__name__}.")
        if "message" in payload:
            raise ValueError(f"Frankfurter returned an error: {payload['message']}")
        for key in ("base", "rates"):
            if key not in payload:
                raise ValueError(f"Response is missing the '{key}' key.")
        if not payload["rates"]:
            raise ValueError("The 'rates' object is empty - no dates matched.")

    elif api_option == "C":
        if not isinstance(payload, dict):
            raise ValueError(f"Expected a dict, got {type(payload).__name__}.")
        if payload.get("type") != "FeatureCollection":
            raise ValueError(
                f"Expected a GeoJSON FeatureCollection, got type={payload.get('type')!r}."
            )
        if not payload.get("features"):
            raise ValueError("No features returned - no events matched the query.")
        metadata_status = payload.get("metadata", {}).get("status")
        if metadata_status is not None and metadata_status != 200:
            raise ValueError(f"USGS reported status {metadata_status} in the metadata.")

    else:
        raise ValueError(f"Unknown API option: {api_option!r}")

    return payload

In [ ]:
# TODO 21 - the validator must pass the good payload AND reject the bad one.
# A validator that never rejects anything is worse than none, because it gives
# false confidence.
print("Good payload:")
checked = check_api_payload(payload, CHOSEN_API)
print("  accepted, returned a", type(checked).__name__,
      "of", len(checked), "items")
print(f"  {len(checked[1])} records received")

print("\nBad payload (from failure 3):")
try:
    check_api_payload(bad_payload, CHOSEN_API)
    print("  PROBLEM: the bad payload was accepted. The check is not working.")
except ValueError as error:
    print("  correctly rejected with:", error)

### What failure looks like - model answer

| Failure | Exception or status | Message | How to handle it in production |
|---|---|---|---|
| Wrong host | `RuntimeError` from the `URLError` branch, **no HTTP status** | `Could not reach the server ...: [Errno -3] Temporary failure in name resolution` | Retry with backoff - this is often a transient network problem rather than a bug. Alert only after several failures. |
| Wrong path | `RuntimeError` from the `HTTPError` branch with a 4xx status, **or** a 200 with an error document, depending on how the path is malformed | `The server refused ... with HTTP status 404` | Do not retry. A 4xx is your mistake, and retrying will fail identically. Fail loudly so the URL gets fixed. |
| Wrong parameter | **HTTP 200**, valid JSON, error message in the body | `[{"message": [{"id": "120", "key": "Invalid value", "value": "The provided parameter value is not valid"}]}]` | Validate the response *shape* after every request, before using it. This is the only one of the three that a status check cannot catch. |

**1. Hardest to notice in a nightly script.** The wrong parameter, by a wide margin. The other two raise exceptions, so the script stops and somebody is alerted. The third returns 200 and valid JSON, so a script that only checks the status code proceeds happily. Worse, if the failure is *partial* - a valid indicator but one mistyped country code - the script produces a table that is the right shape, with the right columns and plausible numbers, and is simply missing a country. Nothing looks wrong. Charts regenerate every morning and the omission can persist indefinitely.

**2. Where the code would have failed without the check.** In `flatten_response()`, at `payload[1]`, with `IndexError: list index out of range` - because the error response is a list of one item, not two. That message says nothing about indicator codes, nothing about the API, and points at the flattening function, which is not where the problem is. A student debugging it would reasonably start by rewriting correct code. `check_api_payload()` instead fails at the point of the actual mistake and names it.

**3. What 200 does not mean.** It means the HTTP transaction succeeded: the request was well-formed, it reached the server, and the server sent a response it considers a successful reply. It does **not** mean the response contains the data you asked for, that your parameters were valid, that the result is complete rather than the first page, or that the body is even the format you expected. HTTP status describes the transport, not the content.

> **Marking note - Part 4.** This part carries the most marks and is the most skipped. Look for:
>
> - **All three failures actually caused**, with output in the notebook. Students often describe them instead of running them. No output, no marks.
> - **`check_api_payload()` demonstrated on both a good and a bad payload.** A validator shown only against valid data is untested. TODO 21 asks for both.
> - **Understanding that failure 3 is different in kind**, not just a third example. A student who wrote "it also gave an error" has missed the entire point of the part.
>
> On failure 2, accept either outcome. The World Bank's behaviour depends on how the path is malformed, and a student who noticed that their "wrong path" also returned 200 with an error document has observed something real - credit it.

## Part 5 - Explore the structure

**TODOs 22 to 25.**

In [ ]:
def describe_json_structure(obj, name="payload", max_depth=2, _depth=0):
    """Print a readable summary of a nested JSON structure."""
    # TODO 22
    pad = "  " * _depth

    if isinstance(obj, dict):
        print(f"{pad}{name}: dict with {len(obj)} keys")
        if _depth < max_depth:
            for key, value in obj.items():
                describe_json_structure(value, str(key), max_depth, _depth + 1)

    elif isinstance(obj, list):
        print(f"{pad}{name}: list of {len(obj)} items")
        # Describe only the FIRST item. 1,800 items of the same shape teach
        # nothing that the first one does not.
        if obj and _depth < max_depth:
            describe_json_structure(obj[0], f"{name}[0]", max_depth, _depth + 1)

    else:
        preview = repr(obj)
        if len(preview) > 60:
            preview = preview[:57] + "..."
        print(f"{pad}{name}: {type(obj).__name__} = {preview}")

In [ ]:
# TODO 23
print("STRUCTURE OF THE RESPONSE")
print("=" * 60)
describe_json_structure(payload, "payload", max_depth=3)

In [ ]:
metadata, records = payload[0], payload[1]

# TODO 24
print("Records received:", len(records))
print("\nKeys on a single record:", list(records[0].keys()))
print("\nOne complete record:")
print(json.dumps(records[0], indent=2))

# TODO 25 - the metadata, which is not the data but tells us how complete it is.
print("\nMetadata (not the data itself):")
for key, value in metadata.items():
    print(f"  {key:14} {value}")
print("\n  'lastupdated' is the field to quote if somebody asks how current")
print("  this data is. 'total' is what tells us whether we have all of it.")

### The shape of the response - model answer

```text
payload                        list of 2 items
├── [0] metadata               dict
│     page, pages, per_page, total, sourceid, lastupdated
└── [1] records                list of 72 items
      └── each record          dict
            ├── indicator      dict  -> {"id": "SP.DYN.LE00.IN", "value": "Life expectancy at birth, total (years)"}
            ├── country        dict  -> {"id": "GH", "value": "Ghana"}
            ├── countryiso3code str   -> "GHA"
            ├── date           str   -> "2023"      <- a YEAR, stored as text
            ├── value          float -> 65.4979     <- or None where absent
            ├── unit           str   -> ""          <- empty for this indicator
            ├── obs_status     str   -> ""
            └── decimal        int   -> 1
```

**2. How many records, and is that all of them?** 72 records - 3 countries times 24 years. The metadata reports `total: 72`, `page: 1`, `pages: 1`, so this is the complete result. It is complete because `per_page` was set to 500, comfortably above the total. With the World Bank's default of 50, this same query would have returned 50 records across 2 pages and Part 10 would have had real work to do.

**3. Nested fields.** `indicator` and `country` are both objects, each with `id` and `value`. A plain `pd.DataFrame(records)` would put whole dictionaries inside those two cells, which is why `pd.json_normalize()` is the right tool - it produces `indicator.value` and `country.value` as ordinary columns.

**4. The currency field for "how current is this?"** `lastupdated` in the metadata, which gives the date the World Bank last revised this indicator's data. Note this is the revision date of the *series*, not of any individual observation, and it is distinct from `date`, which is the year an observation refers to. Confusing the two is easy and consequential.

> **Marking note - Part 5.** `describe_json_structure()` must not print every item of a long list. Test it on a student's Option C payload if they chose it: 1,778 earthquakes printed in full is thousands of lines and shows they did not read the docstring. Question 4 rewards anyone who distinguished `lastupdated` from `date`.

## Part 6 - Flatten into a tidy table

**TODOs 26 to 29.**

In [ ]:
def flatten_response(payload, api_option):
    """Turn a nested API payload into a tidy DataFrame."""
    # TODO 26
    if api_option == "A":
        # Records live at payload[1]; json_normalize flattens the nested
        # indicator and country objects into dotted column names.
        return pd.json_normalize(payload[1])

    if api_option == "B":
        # The dates are dictionary KEYS, so they become the index, then a column.
        wide = pd.DataFrame.from_dict(payload["rates"], orient="index")
        wide.index.name = "date"
        wide = wide.reset_index()
        # Wide (one column per currency) to long (tidy) form.
        return wide.melt(id_vars="date", var_name="currency", value_name="rate")

    if api_option == "C":
        # Deeply nested; json_normalize gives properties.* and geometry.*
        return pd.json_normalize(payload["features"])

    raise ValueError(f"Unknown API option: {api_option!r}")

In [ ]:
# TODO 27
flat_df = flatten_response(payload, CHOSEN_API)
print("Shape:", flat_df.shape)
print("Columns:", list(flat_df.columns))

# TODO 28
print("\nFirst five rows:")
print(flat_df.head().to_string(index=False))
print("\nTypes:")
print(flat_df.dtypes.to_string())

# TODO 29 - confirm nothing nested survived.
still_nested = flat_df.map(lambda cell: isinstance(cell, (dict, list))).any()
print("\nColumns still holding a dict or list:")
print(still_nested[still_nested].to_string() if still_nested.any()
      else "  none - flattening is complete")

> **Marking note - Part 6.** For Option A the giveaway of a correct solution is columns named `indicator.value` and `country.value`. A student who wrote `pd.DataFrame(payload[1])` will have dictionaries sitting in cells, and TODO 29 should have caught it - if their notebook shows nested columns and they continued anyway, they skipped their own check.
>
> For **Option B**, the common failure is stopping at wide format. `pd.DataFrame(payload["rates"])` without `orient="index"` transposes the table so currencies become rows and dates become columns, which then produces a chart with one line per *date*. The `.melt()` to long form is what the tidy requirement asks for.
>
> For **Option C**, look for `properties.time` converted with `unit="ms"`. Without it every date lands in January 1970, and a student who charted that would have a line chart spanning a few hours in 1970 - obvious if they looked at the axis, and easy to miss if they did not.

## Part 7 - Clean it, then save it

**TODOs 30 to 40.**

In [ ]:
def clean_api_data(df, api_option):
    """Return a cleaned copy of the flattened API data."""
    df = df.copy()

    if api_option == "A":
        # TODO 30 - clear names carrying the unit.
        df = df.rename(columns={
            "country.value": "country",
            "countryiso3code": "country_code",
            "indicator.value": "indicator_name",
            "date": "year",
            "value": "life_expectancy_years",
        })

        # TODO 31 - "date" arrives as TEXT even though it is a year.
        df["year"] = df["year"].astype(int)
        df["life_expectancy_years"] = df["life_expectancy_years"].astype(float)

        # TODO 32 - keep what the analysis needs, in reading order.
        df = df[["country", "country_code", "year", "life_expectancy_years"]]

        # TODO 33 - Missing values: the World Bank sends JSON null for a year
        # with no estimate, which pandas holds as NaN. These are dropped, not
        # filled. Filling with 0 would claim a life expectancy of zero years;
        # interpolating would invent an observation and put it on the chart
        # where a reader would take it for a measurement.
        missing = int(df["life_expectancy_years"].isna().sum())
        if missing:
            print(f"Dropping {missing} row(s) with no published estimate.")
        df = df.dropna(subset=["life_expectancy_years"])

        # TODO 34 - the question is about change, so derive it.
        df = df.sort_values(["country", "year"])
        df["years_gained_since_2000"] = (
            df["life_expectancy_years"]
            - df.groupby("country")["life_expectancy_years"].transform("first")
        ).round(2)

        # TODO 35
        return df.sort_values(["country", "year"]).reset_index(drop=True)

    # DELIBERATE, not an oversight: this reference solves Option A. A STUDENT
    # notebook must have no NotImplementedError left anywhere.
    raise NotImplementedError(
        "This reference solves Option A. Options B and C follow the same "
        "structure with different column names - see the marking note below."
    )


def save_clean_data(df, destination):
    """Save a cleaned DataFrame to CSV and verify it reads back correctly."""
    # TODO 36
    destination = Path(destination)
    df.to_csv(destination, index=False)
    reloaded = pd.read_csv(destination)
    return {
        "path": str(destination),
        "rows": int(df.shape[0]),
        "columns": int(df.shape[1]),
        "bytes": destination.stat().st_size,
        "round_trip_ok": bool(reloaded.shape == df.shape),
    }

In [ ]:
# TODO 37
clean_df = clean_api_data(flat_df, CHOSEN_API)

# TODO 38
print("Shape before cleaning:", flat_df.shape)
print("Shape after cleaning: ", clean_df.shape)
print("\nColumns:", list(clean_df.columns))
print("\nTypes:")
print(clean_df.dtypes.to_string())
print("\nMissing values:", int(clean_df.isna().sum().sum()))

# TODO 39
print("\nRanges:")
print(f"  year:                  {clean_df['year'].min()} to {clean_df['year'].max()}")
print(f"  life_expectancy_years: {clean_df['life_expectancy_years'].min():.2f} to "
      f"{clean_df['life_expectancy_years'].max():.2f}")
print(f"  countries:             {sorted(clean_df['country'].unique())}")
print("\nFirst five rows:")
print(clean_df.head().to_string(index=False))

# TODO 40
CLEAN_PATH = CLEAN_DIR / "life_expectancy_ghana_kenya_senegal_2000_2023.csv"
clean_record = save_clean_data(clean_df, CLEAN_PATH)
print("\n" + json.dumps(clean_record, indent=2))

### Cleaning decisions - model answer

| Decision | What I did | Why |
|---|---|---|
| Column names | `country.value`→`country`, `date`→`year`, `value`→`life_expectancy_years` | `value` says nothing about what is being measured or in what unit; the dotted names are awkward to type and invite mistakes |
| Data types | `year` from text to `int`, measure to `float` | The API sends the year as a **string** because JSON has no year type; leaving it as text sorts "2010" before "9" and breaks any numeric axis |
| Columns dropped | `indicator.id`, `unit`, `obs_status`, `decimal`, `country.id` | `unit` and `obs_status` are empty strings for this indicator, and the indicator is constant across every row, so it belongs in the manifest rather than repeated 72 times |
| Missing values | Dropped, never filled | A JSON `null` here means the World Bank published no estimate for that country-year. Filling with 0 asserts a life expectancy of zero; interpolating invents a measurement |
| Derived column | `years_gained_since_2000` | The question asks which country *gained* most, and a level column cannot answer that directly |

**Plausibility.** The values run from roughly 51 to 69 years, which is credible for these three countries over 2000-2023. A value below 20 or above 100 would indicate a unit problem or a wrong indicator code. The year range is exactly 2000-2023 as requested, and all three countries are present with 24 observations each - so nothing was silently dropped.

**Missing values.** None in this extract, because these three indicators are well covered for these countries. Had there been any, dropping them leaves a genuine gap in the line chart, which is the honest representation: matplotlib draws a break where data is absent, and a reader can see that something is missing. Filling with zero would instead have drawn a dramatic plunge to the x-axis and back, which every reader would interpret as a catastrophe rather than as an absence of data.

> **Marking note - Part 7.** The `astype(int)` on the year is the detail that catches people out - the World Bank sends it as a string and everything appears to work until an axis sorts alphabetically. Check `clean_df.dtypes` in their output rather than trusting the code.
>
> On missing values, the reasoning is what earns the marks. Students who wrote `fillna(0)` should be shown what it does to the chart: it is not a neutral default, it is a claim.

## Part 8 - Analyse

**TODOs 41 to 45.**

In [ ]:
def summarise_groups(df, group_column, value_column):
    """Summarise the measure for each group."""
    # TODO 41 - "first" and "last" rely on the sort done in cleaning.
    summary = df.groupby(group_column, as_index=False).agg(
        observations=(value_column, "count"),
        first_value=(value_column, "first"),
        last_value=(value_column, "last"),
        mean_value=(value_column, "mean"),
        min_value=(value_column, "min"),
        max_value=(value_column, "max"),
    )
    summary["change"] = summary["last_value"] - summary["first_value"]
    summary["percent_change"] = (
        summary["change"] / summary["first_value"].where(summary["first_value"] != 0)
        * 100
    )
    return (summary.sort_values("last_value", ascending=False)
                   .round(2)
                   .reset_index(drop=True))


def rank_in_period(df, group_column, value_column, period_column=None,
                   period=None, n=10):
    """Return the top n groups by the measure, optionally within one period."""
    # TODO 42
    subset = df
    if period_column is not None and period is not None:
        subset = subset[subset[period_column] == period]
    return (subset.sort_values(value_column, ascending=False)
                  .head(n)[[group_column, value_column]]
                  .reset_index(drop=True))

In [ ]:
VALUE_COLUMN = "life_expectancy_years"

# TODO 43
group_summary = summarise_groups(clean_df, "country", VALUE_COLUMN)
print("Summary by country, 2000-2023:")
print(group_summary.to_string(index=False))

# TODO 44 - the period is calculated, not typed, so this survives a data update.
latest_year = int(clean_df["year"].max())
ranking = rank_in_period(clean_df, "country", VALUE_COLUMN,
                         period_column="year", period=latest_year, n=10)
print(f"\nRanking in {latest_year}:")
print(ranking.to_string(index=False))

# TODO 45 - the answer as a sentence, with the numbers embedded.
biggest = group_summary.sort_values("change", ascending=False).iloc[0]
print(f"\n{biggest['country']} gained the most: "
      f"{biggest['first_value']:.1f} to {biggest['last_value']:.1f} years, "
      f"a gain of {biggest['change']:.1f} years ({biggest['percent_change']:.1f}%) "
      f"between 2000 and {latest_year}.")

for _, row in group_summary.sort_values("change", ascending=False).iterrows():
    print(f"  {row['country']:<10} {row['first_value']:.1f} -> {row['last_value']:.1f}  "
          f"(+{row['change']:.1f} years, +{row['percent_change']:.1f}%)")

## Part 9 - Two charts

**TODOs 46 to 52.**

In [ ]:
def plot_line_over_time(df, group_column, time_column, value_column,
                        value_label, time_label, title, source_note,
                        save_path=None):
    """Draw a line chart of the measure over time, one line per group."""
    fig, ax = plt.subplots()

    # TODO 46 - sort within each group or the line zig-zags backwards.
    for group_name, group in df.groupby(group_column):
        group = group.sort_values(time_column)
        ax.plot(group[time_column], group[value_column],
                marker="o", markersize=3, linewidth=2, label=str(group_name))

    # TODO 47
    ax.set_title(title)
    ax.set_xlabel(time_label)
    ax.set_ylabel(value_label)
    ax.legend(title=group_column.replace("_", " ").title())

    # TODO 48
    ax.grid(True, alpha=0.3)
    fig.text(0.99, -0.03, source_note, ha="right", fontsize=8, color="grey")
    fig.tight_layout()
    if save_path is not None:
        fig.savefig(save_path, dpi=150, bbox_inches="tight")
    return ax


def plot_bar_comparison(df, category_column, value_column, value_label,
                        title, source_note, save_path=None):
    """Draw a bar chart comparing the measure across categories."""
    fig, ax = plt.subplots()

    # TODO 49
    bars = ax.bar(df[category_column].astype(str), df[value_column],
                  color="#2980b9", edgecolor="white")
    ax.bar_label(bars, fmt="%.1f", fontsize=9, padding=2)

    # TODO 50
    ax.set_title(title)
    ax.set_xlabel(category_column.replace("_", " ").title())
    ax.set_ylabel(value_label)
    if df[category_column].astype(str).str.len().max() > 10:
        plt.setp(ax.get_xticklabels(), rotation=30, ha="right")
    ax.grid(True, axis="y", alpha=0.3)
    ax.set_axisbelow(True)
    fig.text(0.99, -0.03, source_note, ha="right", fontsize=8, color="grey")
    fig.tight_layout()
    if save_path is not None:
        fig.savefig(save_path, dpi=150, bbox_inches="tight")
    return ax

In [ ]:
# The source note is built from request_details, so it reports the date the data
# was actually retrieved rather than a date typed by hand.
SOURCE_NOTE = (f"Source: World Bank Indicators API ({INDICATOR_CODE}), "
               f"retrieved {request_details['retrieved_utc'][:10]}; "
               f"series last updated {metadata['lastupdated']}")

# TODO 51 - the title is written after reading group_summary, so it can state
# what actually happened rather than describing the chart's mechanics.
plot_line_over_time(
    clean_df, "country", "year", VALUE_COLUMN,
    value_label=INDICATOR_LABEL,
    time_label="Year",
    # The title states the finding, and was written AFTER reading
    # group_summary - which is why it names Senegal rather than the country
    # the Part 1 expectation guessed at.
    title="Senegal pulled away from Ghana and Kenya in life expectancy "
          "after 2000",
    source_note=SOURCE_NOTE,
    save_path=CHART_DIR / "life_expectancy_trend_line.png",
)
plt.show()

In [ ]:
# TODO 52
plot_bar_comparison(
    ranking, "country", VALUE_COLUMN,
    value_label=INDICATOR_LABEL,
    title=f"Life expectancy at birth by country, {latest_year}",
    source_note=SOURCE_NOTE,
    save_path=CHART_DIR / "life_expectancy_ranking_bar.png",
)
plt.show()

### Reading the charts - model answers

1. **The line chart** shows all three countries improving, but not together. The three lines start bunched within 2.0 years of each other in 2000 - Ghana 58.1, Senegal 57.6, Kenya 56.1 - and Senegal then separates upward, reaching 68.7 years by 2023 for a gain of 11.1 years. Ghana and Senegal both climb steadily; Kenya's line is the flattest after 2010, gaining only 2.7 years from 2010 to 2023 against 4.8 in the decade before. All three lines dip visibly in 2020 and 2021.
2. **The bar chart** shows the 2023 ranking - Senegal 68.7 years, Ghana 65.5, Kenya 63.6 - a spread of 5.1 years. That is more than twice the 2.0-year spread of 2000, so the three countries **diverged** over this period rather than converging.
3. **Why each chart type.** Year is continuous and ordered, so a line correctly implies a sequence and direction, and lets the reader see the *shape* of each country's path. Country is categorical with no natural order, so a line between countries would draw a trend that does not exist; bars compare magnitudes without implying progression.
4. **The feature worth explaining** is the dip in 2020 and 2021, which appears in all three countries and is far deeper in Kenya: Kenya falls 1.7 years from 62.9 in 2019 to 61.2 in 2021, against 0.8 years for Senegal and 0.2 for Ghana. All three recover by 2022. The timing points strongly at COVID-19 mortality, and the fact that the size differs threefold between neighbouring countries is the interesting part - it suggests real differences in how the pandemic and its recorded mortality played out. This indicator cannot establish any of that: confirming it would need excess-mortality or cause-of-death data for the same years, and a student should say so rather than asserting COVID as the proven cause.

   Worth noting too that this dip is only visible on the line chart. The 2023 ranking on its own gives no hint that anything happened in 2020, which is a concrete illustration of why the assignment asks for both charts.

> **Marking note - Part 9.** Question 4 is where the marks are, and there are two good answers available in this data: the 2020-21 dip, and the divergence between the three countries. Either earns full credit if the student describes it with numbers and says what extra data would be needed to explain it.
>
> The answer to reject is a mechanical description - "the line chart shows life expectancy going up" - which restates the chart rather than reading it. Also correct, in feedback, any student who asserts COVID-19 as the established cause of the dip: the timing is strong evidence and this dataset does not confirm it, and the difference between those two statements is most of what makes analysis trustworthy.
>
> Note that the dip is invisible on the bar chart and the divergence is invisible on the line chart unless the reader compares the endpoints deliberately. Students who used the two charts against each other have understood why both were required.

## Part 10 - Pagination and completeness

**TODO 53.**

In [ ]:
# TODO 53 - Option A: check we received everything, do not assume it.
print("Pagination metadata:")
for key in ("page", "pages", "per_page", "total"):
    print(f"  {key:10} {metadata[key]}")

received = len(records)
total = int(metadata["total"])
print(f"\nRecords received: {received}")
print(f"API says total:   {total}")
print(f"Complete:         {received == total}")

if received != total:
    print("\nINCOMPLETE - fetching the remaining pages.")
    all_records = list(records)
    for page in range(2, int(metadata["pages"]) + 1):
        page_params = dict(MY_PARAMS, page=page)
        page_payload, _ = fetch_json_urllib(MY_ENDPOINT, page_params)
        check_api_payload(page_payload, CHOSEN_API)
        all_records.extend(page_payload[1])
    print(f"After paging: {len(all_records)} records, complete:",
          len(all_records) == total)
else:
    print("\nComplete because per_page=500 exceeds the total of 72.")
    print("With the World Bank's DEFAULT per_page of 50, the same query would")
    print("have returned 50 records across 2 pages. The demonstration below")
    print("shows what that looks like.")

# Deliberately request with a small page size to show the problem existing.
print("\n" + "-" * 60)
print("DEMONSTRATION: the same query with per_page=10")
small_params = dict(MY_PARAMS, per_page=10)
small_payload, _ = fetch_json_urllib(MY_ENDPOINT, small_params)
small_metadata, small_records = small_payload[0], small_payload[1]
print(f"  received {len(small_records)} of {small_metadata['total']} records, "
      f"page {small_metadata['page']} of {small_metadata['pages']}")
print("  A script that stopped here would be analysing "
      f"{len(small_records) / int(small_metadata['total']) * 100:.0f}% of the data")
print("  and nothing about the result would look wrong.")

# And the fix: loop until every page has been collected.
collected = []
for page in range(1, int(small_metadata["pages"]) + 1):
    page_payload, _ = fetch_json_urllib(MY_ENDPOINT, dict(small_params, page=page))
    check_api_payload(page_payload, CHOSEN_API)
    collected.extend(page_payload[1])
print(f"  After looping over all {small_metadata['pages']} pages: "
      f"{len(collected)} records, complete: {len(collected) == total}")

### Completeness - model answers

1. **How many exist, how many do I have.** The metadata reports `total: 72` and I received 72 records, across `pages: 1`. The result is complete.
2. **Why they match, and how I would have known otherwise.** They match because I set `per_page=500`, well above the total. Had they differed, `metadata["pages"]` would have been greater than 1, and the check comparing `len(records)` with `metadata["total"]` would have printed `Complete: False`. The demonstration above forces exactly this case with `per_page=10`: 10 records of 72, across 8 pages, and the loop that follows recovers all 72. **Raising `per_page` is not a general fix** - it works only while the total stays below the limit, so the loop is what you ship.
3. **How a script could be wrong for months.** Nothing about page one is malformed. It has the right columns, the right types, plausible values, and no missing data - it is simply a subset. Downstream code cannot detect this, because a smaller valid table is indistinguishable from a complete one without the metadata. Charts regenerate every morning and look fine. The failure surfaces only when somebody compares a published figure with the source by hand, which may be never. This is why the pagination fields are checked on every request rather than inspected once during development.
4. **Two things that reduce requests.** First, the raw response is saved to `data/raw/` and re-read from disk, so re-running cells does not re-request - the whole notebook can be re-executed for marking without touching the API. Second, `per_page=500` fetches everything in one request instead of the eight that the default page size would have needed.

> **Marking note - Part 10.** The distinction that matters is between *checking* completeness and *asserting* it. A student who wrote "I received all the data" with no comparison against `total` has asserted it. The metadata comparison must appear in their output.
>
> For **Option B**, the equivalent answer is that Frankfurter returns roughly 250 dates for a 365-day range, because the ECB publishes on working days only. A student who reported this as "missing data" has misread it - the absence is a property of the source, not a gap in the retrieval, and the honest treatment is to say so in the limitations. For **Option C**, the check is `metadata.count` against the 20,000-event cap.

## Part 11 - Manifest

**TODOs 54 to 57.**

In [ ]:
def build_manifest(api, request_details, clean_record, question, cleaning_notes,
                   limitations, grain):
    """Assemble a manifest describing this gathering exercise."""
    # TODO 54 - note that no key value appears anywhere. A manifest records
    # WHICH key was used, never its value.
    return {
        "created_utc": datetime.now(timezone.utc).isoformat(),
        "research_question": question,
        "api": {
            "name": api["name"],
            "endpoint": request_details["url"].split("?")[0],
            "documentation": api["documentation"],
            "parameters": request_details["params"],
            "status_code": request_details["status"],
            "retrieved_utc": request_details["retrieved_utc"],
            "api_key_required": api["api_key_required"],
            "api_key_env_var": None,
            "indicator_code": INDICATOR_CODE,
            "indicator_label": INDICATOR_LABEL,
            "series_last_updated": metadata["lastupdated"],
            "records_expected": int(metadata["total"]),
            "records_received": len(records),
        },
        "files": {
            "raw": str(Path(request_details["raw_path"]).relative_to(BASE_DIR))
                   if request_details["raw_path"] else None,
            "clean": str(Path(clean_record["path"]).relative_to(BASE_DIR)),
        },
        "grain": grain,
        "cleaning_steps": list(cleaning_notes),
        "known_limitations": list(limitations),
        "tools": {"pandas": pd.__version__, "python": sys.version.split()[0]},
    }

In [ ]:
# TODO 55
cleaning_notes = [
    "Flattened payload[1] with pd.json_normalize, which turned the nested "
    "indicator and country objects into indicator.value and country.value columns.",
    "Renamed country.value to country, countryiso3code to country_code, date to "
    "year, and value to life_expectancy_years so the unit is in the column name.",
    "Cast year from string to int - the API sends it as text because JSON has no "
    "year type - and the measure to float.",
    "Kept only country, country_code, year, and life_expectancy_years, dropping "
    "the constant indicator fields and the empty unit and obs_status columns.",
    "Dropped rows with a null measure rather than filling them, because a JSON "
    "null means no estimate was published and filling it would invent one.",
    "Derived years_gained_since_2000 per country using a groupby transform on "
    "the first value, because the question asks which country gained most.",
    "Sorted by country then year and reset the index.",
]

# TODO 56
limitations = [
    "Life expectancy here is PERIOD life expectancy at birth: it describes what "
    "a newborn would live to if the current year's mortality rates held for "
    "their whole life. It is not a forecast of how long anyone will live, so I "
    "cannot say a child born in Ghana in 2023 will reach 65.",
    "These are national averages that conceal all variation within a country. "
    "They support no claim about differences between regions, cities, or income "
    "groups within Ghana, Kenya, or Senegal.",
    "The World Bank compiles these figures from national statistical systems of "
    "differing quality, and revises them - the series was last updated on the "
    f"date recorded in the manifest ({metadata['lastupdated']}). A figure quoted "
    "from this extract may not match the same figure requested later, which is "
    "why the retrieval date and the raw response are both preserved.",
    "The finding that Senegal gained the most depends on 2000 being the start "
    "year. Kenya's series is at its lowest point of the whole window in 2000, "
    "so a start year a few years earlier or later would change the gap between "
    "the two and could change the ranking of improvement. The ranking of "
    "improvement is therefore a statement about this window, not a durable "
    "property of the countries.",
    "The 2020-21 dip is present in the data but unexplained by it. This "
    "indicator gives no cause of death, so it cannot support any claim about "
    "why life expectancy fell or why Kenya's fall was larger.",
    "Only three countries were requested, chosen by me. Nothing here supports "
    "any claim about West or East Africa as regions.",
]

# TODO 57
manifest = build_manifest(
    API, request_details, clean_record, RESEARCH_QUESTION,
    cleaning_notes, limitations,
    grain="one row per country per year",
)
MANIFEST_PATH = DATA_DIR / "manifest.json"
MANIFEST_PATH.write_text(json.dumps(manifest, indent=2), encoding="utf-8")
print(f"Wrote {MANIFEST_PATH.name}\n")
print(MANIFEST_PATH.read_text(encoding="utf-8"))

> **Marking note - Part 11.** Check that no secret appears. None of the three options needs a key, so this should be automatic - but a student who went beyond the brief to an API that does need one may well have pasted it. If a key is in the notebook or the manifest, tell them to revoke it at the provider immediately and explain that deleting the line is not enough, because the earlier commit still contains it and the repository is public.
>
> The best manifests record `records_expected` alongside `records_received`, which makes the completeness check permanent rather than a one-off print in Part 10.

## Part 12 - Findings - model answer

**1. Answer to the research question.** All three countries gained life expectancy between 2000 and 2023, but unevenly, and Senegal gained the most: it rose from 57.6 years to 68.7, a gain of 11.1 years or 19.2%. Ghana rose from 58.1 to 65.5, a gain of 7.4 years, and Kenya from 56.1 to 63.6, a gain of 7.6 years. The three countries began the period within 2.0 years of one another and ended it 5.1 years apart, so they **diverged**: the improvement was shared, the rate was not. Ghana and Kenya in particular gained almost identical amounts, 7.4 and 7.6 years, and so remained about 2 years apart throughout - the gap between them in 2023 (1.9 years) is essentially the gap it was in 2000 (2.0 years).

**2. The most surprising thing.** That my Part 1 expectation was wrong. I predicted Kenya would gain the most, reasoning that it started lowest and had the most ground to recover after the HIV/AIDS epidemic. Kenya did start lowest, but it gained only 7.6 years - barely more than Ghana's 7.4 - and finished last. Senegal, which I had not singled out at all, gained 11.1 years and finished first.

I am recording that as the finding rather than quietly adjusting the expectation, because the reasoning behind the prediction was not unreasonable and it was still wrong. "Starting low leaves room to improve" sounds like a mechanism and is really just an assumption; whether a country actually improves depends on things this dataset does not contain. The second surprise was the divergence: I had assumed countries at similar levels would converge, and the spread more than doubled.

**3. Four limitations.** As recorded in the manifest: period life expectancy is not a personal forecast, so I cannot say what any individual child will live to; national averages conceal all within-country variation, so I can say nothing about Northern Ghana versus Accra; the series is revised, so a figure quoted today may not match the same request next year, which is why the raw response and retrieval date are preserved; and the ranking of *improvement* depends on 2000 being the start year, since Kenya sits at its lowest point of the window in that year - so which country "improved most" is a statement about my chosen window rather than a durable fact. I also cannot explain the 2020-21 dip from this indicator, only observe it.

**4. API compared with a file download.**

- **More work:** the API, clearly. A CSV download is two lines and a `read_csv`. The API needed a URL built from parameters, a status check, a shape check, a flattening step for nested objects, a string-to-integer conversion for the year, and a pagination check - none of which a prepared CSV requires.
- **More control:** the API, by just as clear a margin. I asked for three countries, one indicator, and 24 years, and received 72 records. The equivalent download would have been a file covering every country and every year, most of which I would discard.
- **For a monthly report:** the API, without hesitation. The parameters are recorded in the manifest, so refreshing the report is re-running the notebook - and `metadata["lastupdated"]` tells me whether anything actually changed. A download workflow means somebody remembering which page they got the file from, whether the URL still works, and whether the file they have is the current version. That is precisely the problem the checksum in Assignment 1 exists to detect, and the API avoids having it.

The general rule I would take from doing both: download when the published file matches the question closely and rarely changes; use the API when the slice matters, when the data is revised, or when the work will be repeated.

**5. Something that went wrong.** My first attempt returned XML instead of JSON and `json.loads` failed with `Expecting value: line 1 column 1 (char 0)` - a message that says nothing about the cause. I had omitted `format=json`, which the World Bank requires and without which it defaults to XML. I found it by printing the first 200 characters of the raw response instead of the parsed payload, which showed `<?xml version="1.0"...` immediately. The lesson is the one behind saving the raw response before parsing: when parsing fails, the raw text tells you why and the exception does not.

> **Marking note - Part 12.** Question 4 is worth reading properly - it is the question that asks whether the student understood the *point* of two assignments rather than just completing them. Look for a reasoned trade-off rather than a preference. "The API was better" and "downloading was easier" are both half-answers; the full answer names the conditions under which each is right.
>
> Question 5, as in Assignment 1, rewards honesty. The `format=json` error above is genuinely the most common one on Option A, and a student who hit it and diagnosed it has learned more than one who did not.
>
> Question 2 deserves a specific look. This reference deliberately records a Part 1 expectation that the data contradicted, and reports it as such. A student whose stated expectation matches their findings suspiciously well may have written the expectation afterwards; that is worth a light question rather than a deduction, but the habit worth praising is the opposite one - a prediction made in advance, kept on the page, and shown to be wrong.

## Marking summary

| Area | Marks |
|---|---:|
| Tutorial understanding | 8 |
| Question and API choice | 8 |
| URL building | 6 |
| Fetch functions | 16 |
| Failure handling | 14 |
| Structure exploration | 8 |
| Flattening | 10 |
| Cleaning | 14 |
| Saving and raw/clean separation | 6 |
| Analysis functions | 10 |
| Charts | 12 |
| Pagination and completeness | 8 |
| Manifest | 6 |
| Findings and limitations | 10 |
| Repository, README, and commits | 8 |
| **Total** | **144** |

### The six failures to watch for

1. **Parsing before saving the raw response.** Check the order of the lines in `fetch_json_*`.
2. **Skipping Part 4.** The three failures must be caused and their output shown, and `check_api_payload()` must be demonstrated against a bad payload as well as a good one. This is 14 marks and the part that most rewards actually doing it.
3. **Asserting completeness instead of checking it.** The comparison against `metadata["total"]` must appear in the output.
4. **The year left as a string** on Option A. Check `dtypes`, not the code.
5. **`unit="ms"` missing** on Option C. Their line chart will span a few hours in 1970.
6. **Stopping at wide format** on Option B, giving one line per date instead of per currency.

### Repository checks

Confirm: the repository is public; `data/raw/` holds the saved responses as received; `data/clean/` holds the tidy table; the notebook shows outputs and both charts; `manifest.json` is complete and contains no secrets; `README.md` is written; and commit messages describe changes. A notebook pushed without being run cannot be marked on its results.

### A note on the two assignments together

Assignment 1 and Assignment 2 answer nearly the same question about the same three countries by two different routes. That is deliberate. A student who completed both should be able to say why the API version is more work and more controllable, and when each is the right choice - which is what Part 12 question 4 asks. If a cohort struggles with that question, it is worth teaching the two assignments as a pair rather than in sequence.